# Figure3a ucsf activity


In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import anndata as ad
from tqdm import tqdm
from functools import partial
from datetime import datetime
import matplotlib.pyplot as plt
from multiprocessing import Pool
from joblib import dump, load
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve
import warnings
warnings.filterwarnings('ignore')

adata = ad.read_h5ad('data/adata_cohort1.h5ad')
print("Full dataset shape:", adata.shape)

valid_sledai_mask = ((adata.obs['sledai_score'] >= 11) |
                     (adata.obs['sledai_score'] == 0))

adata_filtered = adata[valid_sledai_mask].copy()

adata_filtered.obs['highvslow'] = 'low'
adata_filtered.obs.loc[adata_filtered.obs['sledai_score'] >= 11, 'highvslow'] = 'high'

print("Before removing overlapping donors:")
print(f"  Samples with SLEDAI ≥ 11 (high): {sum(adata_filtered.obs['highvslow'] == 'high')}")
print(f"  Samples with SLEDAI = 0 (low):   {sum(adata_filtered.obs['highvslow'] == 'low')}")
print(f"  Total samples: {adata_filtered.shape[0]}")
print(f"  Unique clues_cohort_id: {adata_filtered.obs['clues_cohort_id'].nunique()}")
print(f"\nHighvslow distribution:")
print(adata_filtered.obs['highvslow'].value_counts(dropna=False))

del adata
gc.collect()

donor_labels = adata_filtered.obs.groupby('clues_cohort_id')['highvslow'].apply(set)
overlapping_donors = donor_labels[donor_labels.apply(lambda x: len(x) > 1)].index.tolist()

print(f"\nFound {len(overlapping_donors)} overlapping donors with both high and low samples:")
for d in overlapping_donors:
    subset = adata_filtered.obs[adata_filtered.obs['clues_cohort_id'] == d]
    print(f"  {d}: {len(subset)} samples — {dict(subset['highvslow'].value_counts())}")

adata_filtered = adata_filtered[~adata_filtered.obs['clues_cohort_id'].isin(overlapping_donors)].copy()

print(f"\nAfter removing overlapping donors:")
print(f"  Total samples: {adata_filtered.shape[0]}")
print(f"  Unique clues_cohort_id: {adata_filtered.obs['clues_cohort_id'].nunique()}")
print(f"  High: {sum(adata_filtered.obs['highvslow'] == 'high')}")
print(f"  Low:  {sum(adata_filtered.obs['highvslow'] == 'low')}")

donor_labels_check = adata_filtered.obs.groupby('clues_cohort_id')['highvslow'].apply(set)
assert all(donor_labels_check.apply(len) == 1), "ERROR: overlapping donors still present!"
print("\n✓ Verified: no donor appears in both high and low groups.")

In [ ]:
def donor_based_cv_split(X_train, y_train, adata, n_splits=5, random_state=42):
    train_donors = np.array(
        adata[adata.obs.index.isin(X_train.index)].obs.clues_cohort_id.unique()
    )

    np.random.seed(random_state)
    np.random.shuffle(train_donors)
    fold_size = len(train_donors) // n_splits
    folds = []

    for i in range(n_splits):
        if i == n_splits - 1:
            val_donors = train_donors[i * fold_size:]
        else:
            val_donors = train_donors[i * fold_size:(i + 1) * fold_size]

        train_donors_fold = np.array([d for d in train_donors if d not in val_donors])

        train_indices = X_train.index[
            adata[adata.obs.index.isin(X_train.index)]
            .obs.clues_cohort_id.isin(train_donors_fold)
        ].tolist()
        val_indices = X_train.index[
            adata[adata.obs.index.isin(X_train.index)]
            .obs.clues_cohort_id.isin(val_donors)
        ].tolist()

        folds.append((train_indices, val_indices))

    return folds

def analyze_single_iteration_with_roc(seed, adata, model_type='elastic_net',
                                      C=1.0, l1_ratio=0.5, n_splits=5):
    gc.collect()

    X_filter = adata.to_df(layer="adjusted_fc_over_ag").copy()
    y_filter = (adata.obs['highvslow'] == 'high').astype(int)
    X_filter[np.isnan(X_filter) | np.isinf(X_filter)] = 0

    n_pos = sum(y_filter)
    n_neg = len(y_filter) - n_pos
    class_ratio = n_pos / n_neg
    class_weights = {0: 1, 1: 1 / class_ratio} if class_ratio < 1 else {0: class_ratio, 1: 1}

    cv_folds = donor_based_cv_split(
        X_filter, y_filter, adata, n_splits=n_splits, random_state=seed
    )

    iteration_results, models, roc_data = [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(cv_folds):
        X_fold_train = X_filter.loc[train_idx]
        X_fold_val = X_filter.loc[val_idx]
        train_mask = X_filter.index.isin(train_idx)
        val_mask = X_filter.index.isin(val_idx)
        y_fold_train = y_filter[train_mask]
        y_fold_val = y_filter[val_mask]

        if model_type == 'ridge':
            model = LogisticRegression(
                penalty='l2', solver='lbfgs', C=C,
                class_weight=class_weights, max_iter=100,
                random_state=seed + fold_idx, n_jobs=1)
        elif model_type == 'lasso':
            model = LogisticRegression(
                penalty='l1', solver='liblinear', C=C,
                class_weight=class_weights, max_iter=100,
                random_state=seed + fold_idx)
        elif model_type == 'elastic_net':
            model = LogisticRegression(
                penalty='elasticnet', solver='saga',
                l1_ratio=l1_ratio, C=C,
                class_weight=class_weights,
                max_iter=100, tol=1e-3,
                random_state=seed + fold_idx, n_jobs=1)

        model.fit(X_fold_train, y_fold_train)
        y_pred_proba = model.predict_proba(X_fold_val)[:, 1]

        fpr, tpr, thresholds = roc_curve(y_fold_val, y_pred_proba)
        prec, rec, pr_thresholds = precision_recall_curve(y_fold_val, y_pred_proba)

        iteration_results.append({
            'seed': seed, 'fold': fold_idx,
            'auroc': roc_auc_score(y_fold_val, y_pred_proba),
            'auprc': auc(rec, prec),
            'n_features': np.sum(model.coef_[0] != 0),
            'n_train_samples': len(X_fold_train),
            'n_test_samples': len(X_fold_val),
            'n_train_donors': len(set(
                adata[adata.obs.index.isin(train_idx)].obs.clues_cohort_id)),
            'n_test_donors': len(set(
                adata[adata.obs.index.isin(val_idx)].obs.clues_cohort_id))
        })

        models.append(model)
        roc_data.append({
            'seed': seed, 'fold': fold_idx, 'fpr': fpr, 'tpr': tpr,
            'thresholds': thresholds, 'y_true': y_fold_val,
            'y_pred_proba': y_pred_proba
        })

    feature_names = list(X_filter.columns)
    del X_filter
    gc.collect()

    return iteration_results, models, roc_data, feature_names

def run_batched_cv_analysis(adata, output_dir, model_type='elastic_net',
                            C=1.0, l1_ratio=0.5, n_iterations=20,
                            batch_size=5, n_jobs=5):
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Model Configuration:")
    print(f"Type: {model_type}")
    print(f"C: {C}")
    if model_type == 'elastic_net':
        print(f"l1_ratio: {l1_ratio}")
    print(f"Iterations: {n_iterations}, Batch size: {batch_size}, Jobs: {n_jobs}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    all_results = []
    all_models = {}
    all_roc_data = []
    feature_names = None

    n_batches = (n_iterations + batch_size - 1) // batch_size

    for batch in tqdm(range(n_batches), desc="Processing batches"):
        start_seed = batch * batch_size
        end_seed = min((batch + 1) * batch_size, n_iterations)
        seeds = [42 + i * 1000 for i in range(start_seed, end_seed)]

        print(f"\nBatch {batch+1}/{n_batches}: Processing iterations {start_seed}-{end_seed-1}")

        analyze_func = partial(
            analyze_single_iteration_with_roc,
            adata=adata, model_type=model_type,
            C=C, l1_ratio=l1_ratio, n_splits=5
        )

        with Pool(n_jobs) as pool:
            batch_results = list(pool.imap(analyze_func, seeds))

        for iteration_idx, (seed, results) in enumerate(zip(seeds, batch_results)):
            iter_results, models, roc_data, features = results

            if feature_names is None:
                feature_names = features

            all_results.extend(iter_results)
            for fold_idx, model in enumerate(models):
                all_models[f"seed_{seed}_fold_{fold_idx}"] = model
            all_roc_data.extend(roc_data)

        temp_df = pd.DataFrame(all_results)
        temp_df.to_csv(f'{output_dir}/results_temp.csv', index=False)
        dump(all_models, f'{output_dir}/models_batch_{batch+1}.joblib')
        dump(all_roc_data, f'{output_dir}/roc_data_batch_{batch+1}.joblib')
        gc.collect()

    results_df = pd.DataFrame(all_results)

    print("\nSaving final results...")
    results_df.to_csv(os.path.join(output_dir, 'results.csv'), index=False)
    dump(all_models, os.path.join(output_dir, 'models.joblib'))
    dump(all_roc_data, os.path.join(output_dir, 'roc_data.joblib'))

    with open(os.path.join(output_dir, 'feature_names.txt'), 'w') as f:
        f.write('\n'.join(feature_names))

    config = {
        'model_type': model_type, 'C': C,
        'l1_ratio': l1_ratio if model_type == 'elastic_net' else None,
        'n_iterations': n_iterations, 'n_splits': 5,
        'batch_size': batch_size, 'n_jobs': n_jobs,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    dump(config, os.path.join(output_dir, 'config.joblib'))

    if os.path.exists(f'{output_dir}/results_temp.csv'):
        os.remove(f'{output_dir}/results_temp.csv')
    for b in range(n_batches):
        for ft in ['models', 'roc_data']:
            bf = f'{output_dir}/{ft}_batch_{b+1}.joblib'
            if os.path.exists(bf):
                os.remove(bf)

    print(f"\nResults Summary:")
    print(f"AUROC: {results_df['auroc'].mean():.4f} ± {results_df['auroc'].std():.4f}")
    print(f"AUPRC: {results_df['auprc'].mean():.4f} ± {results_df['auprc'].std():.4f}")
    print(f"Features: {results_df['n_features'].mean():.0f} ± {results_df['n_features'].std():.0f}")
    print(f"Min features: {results_df['n_features'].min()}")
    print(f"Max features: {results_df['n_features'].max()}")

    return {
        'results_df': results_df, 'models': all_models,
        'roc_data': all_roc_data, 'feature_names': feature_names,
        'config': config
    }

In [ ]:
results = run_batched_cv_analysis(
    adata_filtered,
    output_dir='results/classification_activity/results_elasticnet_C1_l1_0.5',
    model_type='elastic_net', C=1, l1_ratio=0.5,
    n_iterations=20, batch_size=20,
    n_jobs=20
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load

def plot_roc_publication_highlow(
    output_dir,
    save_plot=True,
    figsize=(6, 6),
    ci_level=0.95,
    decimal_places=2,
    show_confidence_interval=True,
    show_operating_point=True,
    operating_point_type='youden',
    target_specificity=0.90,
    primary_color="#b1615c",
):

    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['axes.linewidth'] = 1.2
    plt.rcParams['xtick.major.width'] = 1.2
    plt.rcParams['ytick.major.width'] = 1.2

    results_df = pd.read_csv(f'{output_dir}/results.csv')
    roc_data = load(f'{output_dir}/roc_data.joblib')
    config = load(f'{output_dir}/config.joblib')

    fig, ax = plt.subplots(figsize=figsize)
    mean_fpr = np.linspace(0, 1, 1000)

    tprs = []
    for data in roc_data:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
    tprs = np.array(tprs)

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0

    aucs = results_df['auroc'].values
    mean_auc = np.mean(aucs)
    alpha_low = (1 - ci_level) / 2 * 100
    alpha_high = (1 + ci_level) / 2 * 100
    ci_low = np.percentile(aucs, alpha_low)
    ci_high = np.percentile(aucs, alpha_high)

    if show_confidence_interval:
        tprs_lower = np.percentile(tprs, alpha_low, axis=0)
        tprs_upper = np.percentile(tprs, alpha_high, axis=0)
        ax.fill_between(mean_fpr, tprs_lower, tprs_upper,
                        color=primary_color, alpha=0.15, linewidth=0, zorder=2,
                        label=f'{int(ci_level*100)}% CI')

    fmt = f'.{decimal_places}f'
    label = f'PhIP-seq signature: AUC = {mean_auc:{fmt}} ({ci_low:{fmt}}\u2013{ci_high:{fmt}})'
    ax.plot(mean_fpr, mean_tpr, color=primary_color, linewidth=2.5,
            label=label, zorder=4)

    if show_operating_point:
        if operating_point_type == 'youden':
            youden_index = mean_tpr - mean_fpr
            idx = np.argmax(youden_index)
            op_fpr, op_tpr = mean_fpr[idx], mean_tpr[idx]
            op_sensitivity = op_tpr
            op_specificity = 1 - op_fpr
            op_label = f'Youden: Sens={op_sensitivity:{fmt}}, Spec={op_specificity:{fmt}}'
        elif operating_point_type == 'closest_to_01':
            distances = np.sqrt(mean_fpr**2 + (1 - mean_tpr)**2)
            idx = np.argmin(distances)
            op_fpr, op_tpr = mean_fpr[idx], mean_tpr[idx]
            op_sensitivity = op_tpr
            op_specificity = 1 - op_fpr
            op_label = f'Closest to (0,1): Sens={op_sensitivity:{fmt}}, Spec={op_specificity:{fmt}}'
        else:
            target_fpr = 1 - target_specificity
            idx = np.argmin(np.abs(mean_fpr - target_fpr))
            op_fpr, op_tpr = mean_fpr[idx], mean_tpr[idx]
            op_sensitivity = op_tpr
            op_specificity = 1 - op_fpr
            op_label = f'Sens = {op_tpr:{fmt}} at {int(target_specificity*100)}% Spec'

        ax.scatter([op_fpr], [op_tpr], color=primary_color, s=70, zorder=6,
                   edgecolors='white', linewidths=1.5, marker='o')

        text_offset = (-0.12, -0.08) if op_tpr > 0.5 else (0.05, 0.05)
        ax.annotate(op_label,
                    xy=(op_fpr, op_tpr),
                    xytext=(op_fpr + text_offset[0], op_tpr + text_offset[1]),
                    fontsize=10,
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8,
                                    connectionstyle='arc3,rad=0.1'),
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                              edgecolor='gray', alpha=0.9),
                    zorder=7)

    ax.plot([0, 1], [0, 1], color='gray', linewidth=1.5, linestyle='--',
            alpha=0.7, label='Random classifier', zorder=1)

    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('1 \u2212 Specificity', fontsize=14, fontweight='medium')
    ax.set_ylabel('Sensitivity', fontsize=14, fontweight='medium')
    ax.tick_params(axis='both', which='major', labelsize=12, length=5)
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, zorder=0)
    ax.set_aspect('equal', adjustable='box')

    legend = ax.legend(loc='lower right', fontsize=11, framealpha=0.95,
                       edgecolor='gray', fancybox=False)
    legend.get_frame().set_linewidth(0.8)

    plt.tight_layout()

    if save_plot:
        plt.savefig(f'{output_dir}/roc_curve_highlow.pdf',
                    bbox_inches='tight', facecolor='white', edgecolor='none')

    n_folds = config.get('n_splits', 5)
    n_repeats = len(aucs) // n_folds
    print(f"\nValidation: {n_folds}-fold CV \u00d7 {n_repeats} repeats = {len(aucs)} iterations")
    print(f"Mean AUC: {mean_auc:.4f}")
    print(f"95% CI:   [{ci_low:.4f}, {ci_high:.4f}]")
    if show_operating_point:
        print(f"Operating point ({operating_point_type}): Sensitivity={op_sensitivity:.4f}, Specificity={op_specificity:.4f}")

    plt.show()
    return fig, ax

In [ ]:
fig, ax = plot_roc_publication_highlow(
    'results/classification_activity/results_elasticnet_C1_l1_0.5',
    save_plot=True,
    figsize=(6, 6),
    operating_point_type='closest_to_01',
)